# Light Classification Interpretability
This notebook interprets the lightweight classification pipeline outputs. It trains CNN, BiLSTM, BiGRU, and Transformer models, extracts latent representations, and provides global + per-sample interpretability.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import tensorflow as tf
%matplotlib inline

sys.path.insert(0, "../outlier_detection")
from model_utils import set_global_determinism
import config
from model_utils import create_cnn_model, create_lstm_model, create_gru_model, create_transformer_model

# Optional UMAP
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

tf.get_logger().setLevel('ERROR')
np.random.seed(0)

2026-05-29 22:55:23.457481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-29 22:55:23.474504: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-29 22:55:23.474535: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-29 22:55:23.485716: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-29 22:55:28.226131: W tensorflow/compiler/tf


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



In [2]:
import os
import joblib
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

sys.path.insert(0, "../outlier_detection")
import config
from model_utils import create_cnn_model, create_gru_model, create_transformer_model, set_global_determinism

exp_folder = Path(config.DEFAULT_EXP_FOLDER)
out_subdir = "model_interpretation"
exp_paths = sorted([p for p in exp_folder.iterdir() if p.is_dir() and p.name != '.DS_Store'])
filter_key = 'amf_label_amf_important'

data_packages = []

for exp_path in exp_paths:
    # 1. Define and create the output directory INSIDE the loop so it maps to the correct dataset
    out_dir = exp_path / out_subdir
    out_dir.mkdir(parents=True, exist_ok=True) 

    data = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    
    # Initialize the package with an empty dictionary for model paths
    data_package = {
        "dataset_name" : data["dataset_name"][0],
        "dataset" : data["dataset"][0],
        "features_df" : data["kinetic_features"][0],
        "y_well" : data["Y_well"],
        "timestamps" : data["timestamps"],
        "model_paths" : {}
    }

    encoder = LabelEncoder()
    y_full = encoder.fit_transform(data_package["y_well"])

    mask = (data_package["features_df"][filter_key] == 1).fillna(False).values
    X = data_package["dataset"][mask]
    y = y_full[mask]

    X = X.astype(np.float32)
    X = X[..., None]  # (N, T, 1)
    print(f"\n[{data_package['dataset_name']}] X shape: {X.shape}, y shape: {y.shape}")

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
    train_idx, test_idx = next(splitter.split(X, y))
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    input_size = X_train.shape[1]
    output_size = len(np.unique(y))

    # ---- Train models ----
    set_global_determinism(0)
    models = {
        'cnn': create_cnn_model(input_size, output_size),
        'bigru': create_gru_model(input_size, output_size),
        'transformer': create_transformer_model(input_size, output_size),
    }

    epochs_map = {
        'cnn': 1000,
        'bigru': 500,
        'transformer': 500,
    }

    histories = {}
    
    for name, model in models.items():
        print(f'Training {name}...')
        history = model.fit(
            X_train, y_train,
            epochs=epochs_map[name],
            batch_size=512,
            shuffle=True,
            verbose=0,
        )
        histories[name] = history
        print(f"{name} done. val_acc={history.history['accuracy'][-1]:.3f}")

        # 2. Save the model natively to the output directory
        model_filename = f"{name}_{filter_key}_model.keras"
        model_save_path = out_dir / model_filename
        model.save(model_save_path)

        # 3. Add the saved path to the data package
        data_package["model_paths"][name] = str(model_save_path)

    # Append the completed package
    data_packages.append(data_package)

    # Clear VRAM after processing each dataset's models to prevent OOM errors
    import tensorflow as tf
    tf.keras.backend.clear_session()

joblib.dump(data_packages, exp_folder / "model_interpretation.joblib")


[ori_curves] X shape: (10331, 615, 1), y shape: (10331,)


2026-05-29 22:55:55.119130: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-29 22:55:55.120842: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-29 22:55:55.122144: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Training cnn...


2026-05-29 22:55:57.378088: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


cnn done. val_acc=0.927
Training bigru...
bigru done. val_acc=0.833
Training transformer...
transformer done. val_acc=0.819

[ori_curves] X shape: (6822, 568, 1), y shape: (6822,)
Training cnn...
cnn done. val_acc=0.966
Training bigru...
bigru done. val_acc=0.874
Training transformer...
transformer done. val_acc=0.835

[ori_curves] X shape: (10811, 418, 1), y shape: (10811,)
Training cnn...
cnn done. val_acc=0.875
Training bigru...
bigru done. val_acc=0.814
Training transformer...
transformer done. val_acc=0.793

[ori_curves] X shape: (8254, 664, 1), y shape: (8254,)
Training cnn...
cnn done. val_acc=0.813
Training bigru...
bigru done. val_acc=0.740
Training transformer...
transformer done. val_acc=0.699

[ori_curves] X shape: (4001, 757, 1), y shape: (4001,)
Training cnn...
cnn done. val_acc=0.877
Training bigru...
bigru done. val_acc=0.801
Training transformer...
transformer done. val_acc=0.769

[ori_curves] X shape: (10160, 625, 1), y shape: (10160,)
Training cnn...
cnn done. val_ac

FileNotFoundError: [Errno 2] No such file or directory: '/vol/bitbucket/gk225/POC_DDM_dataset/model_interpretation/curve_for_training.joblib'

In [ ]:
import os
import math
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

for exp_path, package in zip(exp_paths, data_packages):
    out_dir = exp_path / "model_interpretation"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n[*] Generating Visualizations for: {package['dataset_name']}")

    # 1. Reconstruct the filtered X and y data
    filter_key = 'amf_label_amf_important'
    mask = (package["features_df"][filter_key] == 1).fillna(False).values
    X = package["dataset"][mask].astype(np.float32)[..., None]
    
    encoder = LabelEncoder()
    y = encoder.fit_transform(package["y_well"])[mask]
    timestamps = package["timestamps"]

    # 2. Load the trained models for this dataset
    models = {}
    for name, m_path in package["model_paths"].items():
        models[name] = tf.keras.models.load_model(m_path)

    if 'cnn' in models:
        print("  -> Plotting CNN Kernels...")
        cnn_model = models['cnn']
        
        # Find the first Conv1D layer
        conv_layers = [l for l in cnn_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
        if not conv_layers:
            conv_layers = [l for l in cnn_model.layers if 'conv' in l.name.lower()]
            
        if conv_layers:
            layer = conv_layers[0]
            kernels = layer.get_weights()[0]                 
            k_size, in_ch, n_filters = kernels.shape
            kern_mean = kernels.mean(axis=1)                 

            # Representative original signal
            orig_curve = np.squeeze(np.mean(X, axis=0))
            if orig_curve.ndim > 1:
                orig_curve = orig_curve.mean(axis=-1)

            t = timestamps if len(timestamps) == orig_curve.shape[0] else np.arange(orig_curve.shape[0])

            # Setup Subplot Grid (e.g., 16 filters = 4x4 grid)
            cols = 4
            rows = math.ceil(n_filters / cols)
            fig_kern, axes_kern = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3))
            axes_kern = axes_kern.flatten()

            for i in range(n_filters):
                ax1 = axes_kern[i]
                kern = kern_mean[:, i]                      
                filtered = np.convolve(orig_curve, kern, mode='valid')
                
                len_f = filtered.shape[0]
                k_center = k_size // 2
                
                if len(t) == orig_curve.shape[0]:
                    start = max(0, k_center)
                    end = start + len_f
                    if end > len(t):  
                        start = max(0, len(t) - len_f)
                        end = start + len_f
                    t_f = t[start:end]
                else:
                    t_f = np.arange(len_f)

                # Plot Original Curve
                ax1.plot(t, orig_curve, color='k', lw=1.2, label='Original')
                ax1.set_ylabel('Original', color='k', fontsize=9)
                ax1.tick_params(axis='y', labelcolor='k', labelsize=8)
                ax1.tick_params(axis='x', labelsize=8)

                # Plot Filtered Curve
                ax2 = ax1.twinx()
                ax2.plot(t_f, filtered, color='C1', lw=1.2, label=f'Filtered F{i}')
                ax2.set_ylabel('Filtered', color='C1', fontsize=9)
                ax2.tick_params(axis='y', labelcolor='C1', labelsize=8)

                ax1.set_title(f'Filter {i}', fontsize=11, fontweight='bold')

                # INSET: Small kernel reference inside the subplot
                # [x, y, width, height] relative to the subplot dimensions
                axin = ax1.inset_axes([0.65, 0.65, 0.30, 0.25])
                axin.plot(np.arange(k_size), kern, color='C2', lw=1.5)
                axin.set_title('Kernel', fontsize=7, pad=2)
                axin.set_xticks([])
                axin.set_yticks([])

            # Turn off empty subplots if n_filters isn't a perfect multiple of `cols`
            for j in range(n_filters, len(axes_kern)):
                axes_kern[j].axis('off')

            fig_kern.suptitle(f"Layer '{layer.name}' Kernels | kernel_size={k_size}: {package['dataset_name']}", fontsize=16, fontweight='bold', y=1.02)
            fig_kern.tight_layout()
            
            kernel_path = out_dir / "cnn_kernel_visualizations.png"
            fig_kern.savefig(kernel_path, dpi=300, bbox_inches='tight', facecolor='white')
            plt.close(fig_kern)

    # Clear memory before processing the next dataset
    tf.keras.backend.clear_session()

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# ---- Latent extraction & attribution helpers ----
def get_latent_model(model):
    # Penultimate layer (before classifier head)
    return tf.keras.Model(model.input, model.layers[-2].output)

def latent_time_attribution(latent_model, x_batch, top_k=5):
    x_tf = tf.convert_to_tensor(x_batch, dtype=tf.float32)
    z0 = latent_model(x_tf)
    latent_dim = int(z0.shape[-1])

    attributions = []
    for dim in range(latent_dim):
        with tf.GradientTape() as tape_dim:
            tape_dim.watch(x_tf)
            z_dim = latent_model(x_tf)[:, dim]
            target = tf.reduce_mean(z_dim)
        grads = tape_dim.gradient(target, x_tf).numpy()   # (batch, time, channels)
        time_imp = np.mean(np.abs(grads), axis=(0, 2))    # (time,)
        attributions.append(time_imp)

    attributions = np.array(attributions)                 # (latent_dim, time)
    top_dims = np.argsort(attributions.mean(axis=1))[::-1][:top_k]
    return attributions, top_dims

# ====================================================================
# MAIN VISUALIZATION LOOP
# ====================================================================
# Assuming `exp_paths` and `data_packages` are defined from your previous loop
top_k = 10  # Number of dimensions to show per model

for exp_path, package in zip(exp_paths, data_packages):
    out_dir = exp_path / "model_interpretation"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n[*] Generating Latent Time Attributions for: {package['dataset_name']}")

    # 1. Reconstruct X, y, and timestamps
    filter_key = 'amf_label_amf_important'
    mask = (package["features_df"][filter_key] == 1).fillna(False).values
    X = package["dataset"][mask].astype(np.float32)[..., None]
    
    # We only need y to ensure a stratified representative batch
    from sklearn.preprocessing import LabelEncoder
    encoder = LabelEncoder()
    y = encoder.fit_transform(package["y_well"])[mask]
    timestamps = package["timestamps"]

    # 2. Load Models
    models = {}
    for name, m_path in package["model_paths"].items():
        models[name] = tf.keras.models.load_model(m_path)
        
    latent_models = {name: get_latent_model(m) for name, m in models.items()}

    # 3. Build Representative Batch (Stratified)
    batch_size = min(512, len(X))
    rng = np.random.default_rng(42)

    if y is not None and len(np.unique(y)) > 1:
        rep_indices = []
        classes = np.unique(y)
        per_class = max(1, batch_size // len(classes))

        for c in classes:
            idxs = np.where(y == c)[0]
            if len(idxs) > 0:
                take = min(per_class, len(idxs))
                rep_indices.extend(rng.choice(idxs, size=take, replace=False).tolist())

        if len(rep_indices) < batch_size:
            remaining = np.setdiff1d(np.arange(len(X)), np.array(rep_indices, dtype=int))
            extra = rng.choice(remaining, size=min(batch_size - len(rep_indices), len(remaining)), replace=False)
            rep_indices.extend(extra.tolist())

        rep_indices = np.array(rep_indices[:batch_size], dtype=int)
    else:
        rep_indices = np.arange(batch_size)

    X_batch = X[rep_indices]
    t = timestamps if len(timestamps) == X_batch.shape[1] else np.arange(X_batch.shape[1])

    # Pre-calculate mean and std curves for the background plots
    mean_curve = np.squeeze(np.mean(X_batch, axis=0))
    std_curve = np.squeeze(np.std(X_batch, axis=0))
    if mean_curve.ndim > 1:
        mean_curve = mean_curve.mean(axis=-1)
        std_curve = std_curve.mean(axis=-1)

    # 4. Setup Grid Figure (Rows = Models, Columns = Top K Dims)
    num_models = len(models)
    fig, axes = plt.subplots(num_models, top_k, figsize=(4.5 * top_k, 3.5 * num_models), sharex=True)
    
    # Ensure axes is a 2D array even if we only have 1 model or top_k=1
    if num_models == 1 and top_k == 1:
        axes = np.array([[axes]])
    elif num_models == 1:
        axes = axes[np.newaxis, :]
    elif top_k == 1:
        axes = axes[:, np.newaxis]

    # 5. Populate Grid
    for row_idx, (model_name, latent_model) in enumerate(latent_models.items()):
        
        # Calculate attributions for this specific model
        attr, top_dims = latent_time_attribution(latent_model, X_batch, top_k=top_k)
        
        for col_idx, dim in enumerate(top_dims):
            ax1 = axes[row_idx, col_idx]
            
            # Plot the raw batch curve characteristics (Left Y-Axis)
            ax1.plot(t, mean_curve, color='k', lw=1.3, alpha=0.7)
            ax1.fill_between(t, mean_curve - std_curve, mean_curve + std_curve, color='k', alpha=0.10)
            ax1.tick_params(axis='y', labelcolor='k', labelsize=8)
            ax1.tick_params(axis='x', labelsize=8)
            
            if col_idx == 0:
                ax1.set_ylabel(f"{model_name.upper()}\nMean Curve", color='k', fontweight='bold')

            # Plot the Attribution gradients (Right Y-Axis)
            ax2 = ax1.twinx()
            ax2.plot(t, attr[dim], color='C1', lw=1.5)
            ax2.tick_params(axis='y', labelcolor='C1', labelsize=8)
            
            if col_idx == top_k - 1:
                ax2.set_ylabel('Attribution', color='C1', fontweight='bold')

            ax1.set_title(f'Top Dim {dim}', fontsize=11)

    # Figure Formatting & Saving
    fig.suptitle(f"Latent Time Attribution (Top {top_k} Dims) | Dataset: {package['dataset_name']}", fontsize=16, fontweight='bold', y=1.02)
    fig.tight_layout()
    
    save_path = out_dir / "latent_time_attribution_grid.png"
    fig.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [✓] Saved grid to {save_path}")

    # Clear memory
    tf.keras.backend.clear_session()

In [ ]:
# ---- Latent extraction helpers ----
def get_latent_model(model):
    # Penultimate layer (before classifier head)
    return tf.keras.Model(model.input, model.layers[-2].output)

latent_models = {name: get_latent_model(m) for name, m in models.items()}

latents = {name: lm.predict(X, verbose=0) for name, lm in latent_models.items()}
for name, z in latents.items():
    print(name, z.shape)

In [ ]:
# ---- Global latent structure ----
def plot_latent_2d(z, title):
    pca = PCA(n_components=2, random_state=0)
    z_pca = pca.fit_transform(z)

    plt.figure(figsize=(6, 5))
    plt.scatter(z_pca[:, 0], z_pca[:, 1], c=y, s=10, cmap='tab10')
    plt.title(f'{title} - PCA')
    plt.tight_layout()
    plt.show()

    tsne = TSNE(n_components=2, perplexity=30, random_state=0, init='pca')
    z_tsne = tsne.fit_transform(z)
    plt.figure(figsize=(6, 5))
    plt.scatter(z_tsne[:, 0], z_tsne[:, 1], c=y, s=10, cmap='tab10')
    plt.title(f'{title} - t-SNE')
    plt.tight_layout()
    plt.show()

    if HAS_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=0)
        z_umap = reducer.fit_transform(z)
        plt.figure(figsize=(6, 5))
        plt.scatter(z_umap[:, 0], z_umap[:, 1], c=y, s=10, cmap='tab10')
        plt.title(f'{title} - UMAP')
        plt.tight_layout()
        plt.show()

for name, z in latents.items():
    plot_latent_2d(z, name)

In [ ]:
models.keys()

In [ ]:
# ---- Saliency and integrated gradients ----
def saliency_map(model, x, class_index=None):
    x_tf = tf.convert_to_tensor(x[None, ...], dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tf)
        preds = model(x_tf)
        if class_index is None:
            class_index = int(tf.argmax(preds[0]).numpy())
        target = preds[:, class_index]
    grads = tape.gradient(target, x_tf).numpy()[0]
    return grads.squeeze(), class_index

def integrated_gradients(model, x, class_index=None, baseline=None, steps=50):
    x_tf = tf.convert_to_tensor(x[None, ...], dtype=tf.float32)
    if baseline is None:
        baseline = tf.zeros_like(x_tf)
    if class_index is None:
        preds = model(x_tf)
        class_index = int(tf.argmax(preds[0]).numpy())

    alphas = tf.linspace(0.0, 1.0, steps + 1)
    total_grads = tf.zeros_like(x_tf)

    for alpha in alphas:
        x_step = baseline + alpha * (x_tf - baseline)
        with tf.GradientTape() as tape:
            tape.watch(x_step)
            preds = model(x_step)
            target = preds[:, class_index]
        grads = tape.gradient(target, x_step)
        total_grads += grads

    avg_grads = total_grads / (steps + 1)
    ig = (x_tf - baseline) * avg_grads
    return ig.numpy()[0].squeeze(), class_index


# ---- Latent dimension to time attribution ----
def latent_time_attribution(latent_model, x_batch, top_k=10):
    x_tf = tf.convert_to_tensor(x_batch, dtype=tf.float32)
    z0 = latent_model(x_tf)
    latent_dim = int(z0.shape[-1])

    attributions = []
    for dim in range(latent_dim):
        with tf.GradientTape() as tape_dim:
            tape_dim.watch(x_tf)
            z_dim = latent_model(x_tf)[:, dim]
            target = tf.reduce_mean(z_dim)
        grads = tape_dim.gradient(target, x_tf).numpy()   # (batch, time, channels)
        time_imp = np.mean(np.abs(grads), axis=(0, 2))    # (time,)
        attributions.append(time_imp)

    attributions = np.array(attributions)                 # (latent_dim, time)
    top_dims = np.argsort(attributions.mean(axis=1))[::-1][:top_k]
    return attributions, top_dims


def class_sample_index_map(y_labels, n_classes):
    """Pick one test sample index per class."""
    out = {}
    for c in range(n_classes):
        idxs = np.where(y_labels == c)[0]
        out[c] = int(idxs[0]) if len(idxs) > 0 else None
    return out


for model_name, model in models.items():

    # infer class count from model output
    n_classes = int(model(tf.convert_to_tensor(X_test[:1], dtype=tf.float32)).shape[-1])
    class_to_idx = class_sample_index_map(y_test, n_classes)

    # 1) Saliency: one figure with n_classes subplots (class-matched samples)
    fig, axes = plt.subplots(n_classes, 1, figsize=(12, 2.6 * n_classes), sharex=False)
    if n_classes == 1:
        axes = [axes]

    for c in range(n_classes):
        ax1 = axes[c]

        # Find samples where true label matches class c and prediction is confident
        class_mask = (y_test == c)
        class_indices = np.where(class_mask)[0]
        
        if len(class_indices) > 0:
            # Get prediction probabilities for this class
            preds = model(tf.convert_to_tensor(X_test[class_indices], dtype=tf.float32)).numpy()
            pred_classes = np.argmax(preds, axis=1)
            pred_confidence = np.max(preds, axis=1)
            
            # Find correctly predicted samples with highest confidence
            correct_mask = (pred_classes == c)
            if np.any(correct_mask):
                correct_indices = class_indices[correct_mask]
                correct_confidence = pred_confidence[correct_mask]
                best_correct_idx = correct_indices[np.argmax(correct_confidence)]
                idx = best_correct_idx
            else:
                idx = class_indices[0]
                
        if idx is None:
            ax1.text(0.5, 0.5, f'No sample found for class {c}', ha='center', va='center')
            ax1.set_title(f'Class {c}')
            ax1.set_axis_off()
            continue

        x_sample = X_test[idx]
        t = timestamps if len(timestamps) == len(x_sample) else np.arange(len(x_sample))
        x_curve = x_sample.squeeze()

        sal, _ = saliency_map(model, x_sample, class_index=c)

        ax1.plot(t, x_curve, color='C0', lw=1.2, label=f'sample (true class={int(y_test[idx])})')
        ax1.set_ylabel('curve', color='C0')
        ax1.tick_params(axis='y', labelcolor='C0')
        ax1.set_title(f'Class {c} | sample_idx={idx}')

        ax2 = ax1.twinx()
        ax2.plot(t, sal, color='C1', alpha=0.8, lw=1.0, label=f'saliency class {c}')
        ax2.set_ylabel(f'sal c{c}', color='C1')
        ax2.tick_params(axis='y', labelcolor='C1')

    fig.suptitle(f'{model_name} - Saliency per class (class-specific samples)')
    fig.tight_layout()
    plt.show()

    # 2) Integrated Gradients: one figure with n_classes subplots (class-matched samples)
    fig, axes = plt.subplots(n_classes, 1, figsize=(12, 2.6 * n_classes), sharex=False)
    if n_classes == 1:
        axes = [axes]

    for c in range(n_classes):
        ax1 = axes[c]
        idx = class_to_idx[c]
        if idx is None:
            ax1.text(0.5, 0.5, f'No sample found for class {c}', ha='center', va='center')
            ax1.set_title(f'Class {c}')
            ax1.set_axis_off()
            continue

        x_sample = X_test[idx]
        t = timestamps if len(timestamps) == len(x_sample) else np.arange(len(x_sample))
        x_curve = x_sample.squeeze()

        ig, _ = integrated_gradients(model, x_sample, class_index=c)

        ax1.plot(t, x_curve, color='C0', lw=1.2)
        ax1.set_ylabel('curve', color='C0')
        ax1.tick_params(axis='y', labelcolor='C0')
        ax1.set_title(f'Class {c} | sample_idx={idx}')

        ax2 = ax1.twinx()
        ax2.plot(t, ig, color='C2', alpha=0.8, lw=1.0)
        ax2.set_ylabel(f'IG c{c}', color='C2')
        ax2.tick_params(axis='y', labelcolor='C2')

    fig.suptitle(f'{model_name} - Integrated Gradients per class (class-specific samples)')
    fig.tight_layout()
    plt.show()

    # 3) Latent attribution + sample curve (class-matched samples, separate y-scale)
    latent_model = latent_models[model_name]

    # Representative batch: stratified across classes when labels are available
    batch_size = min(512, len(X_test))
    rng = np.random.default_rng(42)

    if y_test is not None and len(np.unique(y_test)) > 1:
        rep_indices = []
        classes = np.unique(y_test)
        per_class = max(1, batch_size // len(classes))

        for c in classes:
            idxs = np.where(y_test == c)[0]
            if len(idxs) > 0:
                take = min(per_class, len(idxs))
                rep_indices.extend(rng.choice(idxs, size=take, replace=False).tolist())

        if len(rep_indices) < batch_size:
            remaining = np.setdiff1d(np.arange(len(X_test)), np.array(rep_indices, dtype=int))
            extra = rng.choice(remaining, size=min(batch_size - len(rep_indices), len(remaining)), replace=False)
            rep_indices.extend(extra.tolist())

        rep_indices = np.array(rep_indices[:batch_size], dtype=int)
    else:
        rep_indices = np.arange(batch_size)

    X_batch = X_test[rep_indices]
    attr, top_dims = latent_time_attribution(latent_model, X_batch, top_k=5)

    t = timestamps if len(timestamps) == X_batch.shape[1] else np.arange(X_batch.shape[1])

    mean_curve = np.squeeze(np.mean(X_batch, axis=0))
    std_curve = np.squeeze(np.std(X_batch, axis=0))
    if mean_curve.ndim > 1:
        mean_curve = mean_curve.mean(axis=-1)
        std_curve = std_curve.mean(axis=-1)

    for dim in top_dims:
        fig, ax1 = plt.subplots(figsize=(12, 4))
        ax1.plot(t, mean_curve, color='k', lw=1.3, label='batch mean curve')
        ax1.fill_between(t, mean_curve - std_curve, mean_curve + std_curve, color='k', alpha=0.12)
        ax1.set_ylabel('curve', color='k')
        ax1.tick_params(axis='y', labelcolor='k')
        ax1.set_title(f'Batch latent attribution | sample count={len(rep_indices)}')

        ax2 = ax1.twinx()
    
        ax2.plot(t, attr[dim], lw=1.0, label=f'latent dim {dim}')
        ax2.set_ylabel('latent attr')

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

        fig.suptitle(f'{model_name} - dim {dim} - Latent attribution on one representative batch')
        fig.tight_layout()
        plt.show()

In [ ]:

# pick first conv layer (or adjust if you want a different one)
conv_layers = [l for l in model.layers if isinstance(l, tf.keras.layers.Conv1D)]
if not conv_layers:
    conv_layers = [l for l in model.layers if 'conv' in l.name.lower()]
if not conv_layers:
    raise RuntimeError("No Conv1D layer found in cnn model.")

layer = conv_layers[0]
kernels = layer.get_weights()[0]                 # (kernel_size, in_channels, out_filters)
k_size, in_ch, n_filters = kernels.shape
kern_mean = kernels.mean(axis=1)                 # (kernel_size, n_filters)

# representative original signal: mean across dataset (use X or X_test as desired)
orig_curve = np.squeeze(np.mean(X, axis=0))
if orig_curve.ndim > 1:
    orig_curve = orig_curve.mean(axis=-1)

t = timestamps if len(timestamps) == orig_curve.shape[0] else np.arange(orig_curve.shape[0])

# Optionally limit how many filters to visualize (set `take` in notebook); here honor `take` if present
n_show = n_filters if 'take' not in globals() else min(n_filters, int(take))

for i in range(n_show):
    kern = kern_mean[:, i]                      # (kernel_size,)
    # convolve with 'valid' and align time axis to kernel center
    filtered = np.convolve(orig_curve, kern, mode='valid')
    len_f = filtered.shape[0]
    k_center = k_size // 2
    # compute filtered time axis (try to align to original timestamps)
    if len(t) == orig_curve.shape[0]:
        start = k_center
        end = start + len_f
        if end > len(t):  # fallback if kernel even/edge cases
            start = max(0, len(t) - len_f)
            end = start + len_f
        t_f = t[start:end]
    else:
        t_f = np.arange(len_f)

    plt.figure(figsize=(10, 3.2))
    ax1 = plt.gca()
    ax1.plot(t, orig_curve, color='k', lw=1.2, label='original')
    ax1.set_ylabel('original', color='k')
    ax1.tick_params(axis='y', labelcolor='k')

    ax2 = ax1.twinx()
    ax2.plot(t_f, filtered, color='C1', lw=1.0, label=f'filtered F{i}')
    ax2.set_ylabel('filtered', color='C1')
    ax2.tick_params(axis='y', labelcolor='C1')

    plt.title(f'Layer {layer.name} | filter {i} | kernel_size={k_size}')
    # show kernel as inset (small axis) for reference
    axin = plt.axes([0.78, 0.62, 0.15, 0.25])
    axin.plot(np.arange(k_size), kern, color='C2', lw=1)
    axin.set_title('kernel', fontsize=8)
    axin.set_xticks([])
    axin.set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
# Use CNN explicitly (since `model` may currently point to transformer)
cnn_model = models["cnn"]

# Sample input: mean curve over X
orig_curve = np.squeeze(np.mean(X, axis=0))
if orig_curve.ndim > 1:
    orig_curve = orig_curve.mean(axis=-1)
x_input = orig_curve[None, :, None].astype(np.float32)  # (1, T, 1)

t = timestamps if len(timestamps) == len(orig_curve) else np.arange(len(orig_curve))

# ---- 1) First layer output (input to second layer) ----
# Prefer first Conv1D layer for kernel-wise interpretation
conv_layers = [l for l in cnn_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
if not conv_layers:
    raise RuntimeError("No Conv1D layer found in cnn model.")
first_conv = conv_layers[0]

first_layer_model = tf.keras.Model(inputs=cnn_model.input, outputs=first_conv.output)
first_out = first_layer_model.predict(x_input, verbose=0)[0]  # (T_out, n_filters)

print("First conv layer:", first_conv.name)
print("First conv output shape:", first_out.shape)

# Plot first-layer feature maps (these are passed to the next layer)
n_filters = first_out.shape[-1]
fig, axes = plt.subplots(n_filters, 1, figsize=(12, max(2.2 * n_filters, 4)), sharex=True)
if n_filters == 1:
    axes = [axes]

t_out = np.arange(first_out.shape[0])
for i, ax in enumerate(axes):
    ax.plot(t_out, first_out[:, i], lw=1.0)
    ax.set_ylabel(f"F{i}")
axes[-1].set_xlabel("time index (layer output)")
fig.suptitle(f"{first_conv.name} outputs (input to next layer)")
fig.tight_layout()
plt.show()

# ---- 2) Visualize each kernel/weight ----
w = first_conv.get_weights()
kernels = w[0]  # (kernel_size, in_channels, out_filters)
bias = w[1] if len(w) > 1 else None
k_size, in_ch, out_ch = kernels.shape

fig, axes = plt.subplots(out_ch, 1, figsize=(10, max(2.0 * out_ch, 4)), sharex=True)
if out_ch == 1:
    axes = [axes]

k_idx = np.arange(k_size)
for f, ax in enumerate(axes):
    # If multiple input channels exist, show mean kernel + faint per-channel traces
    if in_ch > 1:
        for c in range(in_ch):
            ax.plot(k_idx, kernels[:, c, f], alpha=0.25, lw=0.8)
        ax.plot(k_idx, kernels[:, :, f].mean(axis=1), lw=1.5, color="k", label="mean over in_ch")
        ax.legend(loc="upper right", fontsize=8)
    else:
        ax.plot(k_idx, kernels[:, 0, f], lw=1.2)
    btxt = f", b={bias[f]:.4f}" if bias is not None else ""
    ax.set_ylabel(f"K{f}{btxt}", fontsize=8)

axes[-1].set_xlabel("kernel index")
fig.suptitle(f"{first_conv.name} kernels")
fig.tight_layout()
plt.show()

# ---- 3) Visualize each kernel output on orig_curve ----
# Using model-computed first layer outputs (includes conv + layer activation)
fig, axes = plt.subplots(out_ch, 1, figsize=(12, max(2.2 * out_ch, 4)), sharex=True)
if out_ch == 1:
    axes = [axes]

for f, ax in enumerate(axes):
    ax.plot(t_out, first_out[:, f], color="C1", lw=1.0)
    ax.set_ylabel(f"out K{f}", fontsize=8)

axes[-1].set_xlabel("time index")
fig.suptitle(f"{first_conv.name} per-kernel outputs for orig_curve")
fig.tight_layout()
plt.show()

In [ ]:
# ---- Latent dimension to time attribution ----
def latent_time_attribution(latent_model, x_batch, top_k=5):
    x_tf = tf.convert_to_tensor(x_batch, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tf)
        z = latent_model(x_tf)
    latent_dim = z.shape[-1]
    attributions = []
    for dim in range(latent_dim):
        with tf.GradientTape() as tape_dim:
            tape_dim.watch(x_tf)
            z_dim = latent_model(x_tf)[:, dim]
            target = tf.reduce_mean(z_dim)
        grads = tape_dim.gradient(target, x_tf).numpy()
        time_imp = np.mean(np.abs(grads), axis=(0, 2))
        attributions.append(time_imp)

    attributions = np.array(attributions)
    top_dims = np.argsort(attributions.mean(axis=1))[::-1][:top_k]
    return attributions, top_dims

latent_model = latent_models['transformer']
attr, top_dims = latent_time_attribution(latent_model, X_test[:128])

plt.figure(figsize=(10, 4))
for dim in top_dims:
    plt.plot(attr[dim], label=f'dim {dim}')
plt.legend()
plt.title('Top latent dims: time attribution')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Case study: TP / FP / FN ----
from sklearn.metrics import confusion_matrix

model = models['transformer']
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:', cm)

tp_idx = np.where((y_test == y_pred) & (y_test == y_test[0]))[0]
fp_idx = np.where((y_test != y_pred) & (y_pred == y_test[0]))[0]
fn_idx = np.where((y_test != y_pred) & (y_test == y_test[0]))[0]

def plot_case(idx, title):
    x = X_test[idx].squeeze()
    plt.figure(figsize=(8, 3))
    plt.plot(x)
    plt.title(title)
    plt.tight_layout()
    plt.show()

if len(tp_idx) > 0:
    plot_case(tp_idx[0], 'TP example')
if len(fp_idx) > 0:
    plot_case(fp_idx[0], 'FP example')
if len(fn_idx) > 0:
    plot_case(fn_idx[0], 'FN example')

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.layers import InputLayer
from tensorflow.keras.utils import plot_model
import visualkeras

def build_cnn_autoencoder(timesteps):
    inputs = layers.Input(shape=(timesteps, 1))
    
    x = layers.Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x = layers.Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    
    x = layers.Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.UpSampling1D(size=2)(x)
    x = layers.Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(x)
    x = layers.UpSampling1D(size=2)(x)
    
    decoded = layers.Conv1D(filters=1, kernel_size=3, activation='linear', padding='same')(x)
    decoded = decoded[:, :timesteps, :] # Safety slice for downsampling division remainder
    
    autoencoder = models.Model(inputs, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder




In [ ]:
from tensorflow.keras.utils import plot_model
import IPython

# Build the model
autoencoder = build_cnn_autoencoder(timesteps=100)

# Generate and display the flowchart inline
plot_model(
    autoencoder, 
    show_shapes=True, 
    show_layer_names=True,
    dpi=96           # Keeps the image from becoming massive in Jupyter
)

In [ ]:
from tensorflow.keras.utils import plot_model
import IPython

# Build the model
autoencoder = build_cnn_autoencoder(timesteps=100)

# Generate and display the flowchart inline
plot_model(
    autoencoder, 
    show_shapes=True, 
    show_layer_names=True,
    dpi=96
)